# Bin2cell: From Visium HD 2 um Bins to Single Cells

This notebook uses [bin2cell](https://github.com/Teichlab/bin2cell) to aggregate Visium HD **2 um bins** into a near **single-cell** resolution expression matrix.

**Idea:** Visium HD's native unit is a 2 um square (bin), not a cell. We first run StarDist nucleus segmentation on both the **H&E morphology image** and a **gene-expression image**, then merge bins that fall inside the same nucleus into one cell.

**Pipeline:** read data -> build H&E image -> destripe -> H&E segmentation -> GEX segmentation -> merge labels -> aggregate to cells -> save.

## Setup and imports
Raise OpenCV's max image-pixel limit (whole-slide H&E images are huge), import scanpy / OpenCV / bin2cell.

In [1]:
import os
os.environ['OPENCV_IO_MAX_IMAGE_PIXELS'] = str(2**32)  # Set to a very large value (4 billion pixels)

import matplotlib.pyplot as plt
import scanpy as sc
import cv2
import bin2cell as b2c
import time

notebook_start = time.time()

## 1. Load raw data

- `path`: Visium HD `square_002um` (2 um bin) output directory
- `source_image_path`: the full-resolution H&E tissue image

`b2c.read_visium` loads both the expression matrix and the high-resolution image; `var_names_make_unique()` resolves duplicate gene names.

In [2]:
path = "/Volumes/Siyuan SSD/ST/skin_TXK6Z4X_A1_lite/binned_outputs/square_002um"
source_image_path = "/Volumes/Siyuan SSD/ST/H1-TXK6Z4X-A1_SK24-001_A1-4-003.tiff"
os.makedirs("stardist", exist_ok=True)

In [3]:
adata = b2c.read_visium(path, source_image_path = source_image_path)
adata.var_names_make_unique()
adata

/Users/siyuanzhao/anaconda3/envs/loom/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/siyuanzhao/anaconda3/envs/loom/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 4919197 × 18085
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

### Basic QC filtering

Drop genes detected in almost no bins (`min_cells=3`) and empty bins with no signal (`min_counts=1`) to reduce downstream computation.

In [4]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_counts=1)
adata

AnnData object with n_obs × n_vars = 3550646 × 17079
    obs: 'in_tissue', 'array_row', 'array_col', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial'

## 2. Set resolution and build the H&E image

`mpp` (microns per pixel) sets the scale for every image step and **must stay consistent** across image generation, segmentation, and label mapping.

`b2c.scaled_he_image` resamples the full-resolution H&E to this resolution and saves it as `stardist/he.tiff` (the StarDist input). It also writes cropped coordinates to `obsm['spatial_cropped_150_buffer']` and the scaled image into `uns['spatial']`.

In [5]:
mpp = 0.5

In [6]:
b2c.scaled_he_image(adata, mpp=mpp, save_path="stardist/he.tiff")

Cropped spatial coordinates key: spatial_cropped_150_buffer
Image key: 0.5_mpp_150_buffer


## 3. Destripe

Visium HD shows systematic row/column intensity differences (striping) due to how the instrument reads the slide. `b2c.destripe` normalizes each bin's total counts to correct this; the result is stored in `obs['n_counts_adjusted']` and used later to render the expression image.

In [7]:
b2c.destripe(adata,adjust_counts=True)

/Users/siyuanzhao/anaconda3/envs/loom/lib/python3.11/site-packages/scipy/sparse/_construct.py:367: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if len(diagonals) == 0 or isscalarlike(diagonals[0]):


## 4. Nucleus segmentation on the H&E image

Run StarDist's H&E pretrained model `2D_versatile_he` on `he.tiff` to segment nuclei, producing a pixel-level label matrix `he.npz` (background = 0, one integer id per nucleus). A lower `prob_thresh` recalls more nuclei.

In [8]:
b2c.stardist(
    image_path="stardist/he.tiff",
    labels_npz_path="stardist/he.npz",
    stardist_model="2D_versatile_he",
    prob_thresh=0.01,
)

Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
Overriding with prob_thresh=0.01
effective: block_size=(4096, 4096, 3), min_overlap=(128, 128, 0), context=(128, 128, 0)


100%|██████████| 16/16 [07:52<00:00, 29.55s/it]


Found 113158 objects


## 5. Map H&E labels onto bins and expand

`insert_labels` looks up each bin's spatial coordinate in `he.npz` and writes the owning nucleus id into `obs['labels_he']` (0 outside any nucleus).

`expand_labels` then grows each nucleus label outward to cover the surrounding cytoplasm, giving `labels_he_expanded` -- transcripts are not confined to the nucleus but spread across the whole cell body.

In [9]:
b2c.insert_labels(adata, 
                  labels_npz_path="stardist/he.npz", 
                  basis="spatial", 
                  spatial_key="spatial_cropped_150_buffer",
                  mpp=mpp, 
                  labels_key="labels_he"
                 )

In [10]:
b2c.expand_labels(adata, 
                  labels_key='labels_he', 
                  expanded_labels_key="labels_he_expanded"
                 )

## 6. Expression-based nucleus segmentation

`grid_image` renders the destriped total counts (`n_counts_adjusted`) into a single-channel grayscale expression image `gex.tiff` (bright where cells are, dark in the gaps). StarDist's fluorescence model `2D_versatile_fluo` then segments this image into `gex.npz`.

This expression-based pass recovers cells that are unclear on H&E but clearly express mRNA, complementing the H&E segmentation.

In [11]:
img = b2c.grid_image(adata, "n_counts_adjusted", mpp=mpp, sigma=5)
cv2.imwrite("stardist/gex.tiff", img)

True

In [12]:
b2c.stardist(image_path="stardist/gex.tiff", 
             labels_npz_path="stardist/gex.npz", 
             stardist_model="2D_versatile_fluo", 
             prob_thresh=0.05, 
             nms_thresh=0.5
            )

Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
Overriding with prob_thresh=0.05
Overriding with nms_thresh=0.5
effective: block_size=(4096, 4096), min_overlap=(128, 128), context=(128, 128)


100%|██████████| 16/16 [12:28<00:00, 46.77s/it]


Found 54124 objects


### Map GEX labels onto bins

Write the `gex.npz` nucleus labels into `obs['labels_gex']`. Note `basis="array"` here (align by bin array coordinates), since the expression image is rendered on the bin grid rather than physical space.

In [13]:
b2c.insert_labels(adata, 
                  labels_npz_path="stardist/gex.npz", 
                  basis="array", 
                  mpp=mpp, 
                  labels_key="labels_gex"
                 )

## 7. Merge the two label sets

`salvage_secondary_labels` uses the expanded H&E labels as the primary source and fills regions H&E missed with the GEX labels, producing the final `labels_joint`. The resulting `cdata.obs['labels_joint_source']` records whether each cell's label came from H&E or GEX.

In [14]:
b2c.salvage_secondary_labels(adata, 
                             primary_label="labels_he_expanded", 
                             secondary_label="labels_gex", 
                             labels_key="labels_joint"
                            )

Salvaged 14851 secondary labels


## 8. Aggregate bins into single cells

`bin_to_cell` sums the expression of all bins sharing the same `labels_joint`, producing a single-cell AnnData (`cdata`) while keeping both `spatial` and `spatial_cropped_150_buffer` coordinates. Each obs row is now one cell, carrying `object_id` (nucleus id) and `bin_count` (how many bins formed the cell).

In [15]:
cdata = b2c.bin_to_cell(adata, labels_key="labels_joint", spatial_keys=["spatial", "spatial_cropped_150_buffer"])

In [16]:
cdata

AnnData object with n_obs × n_vars = 114213 × 17079
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial', 'spatial_cropped_150_buffer'

## 9. Save results

In [17]:
cdata.write_h5ad('skin_TXK6Z4X_A1_2um_b2c.h5ad')

elapsed = time.time() - notebook_start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)

print(f"Total notebook runtime: {hours}h {minutes}m {seconds}s")

Total notebook runtime: 0h 24m 12s
